# 03 - Build Gold Price Review Output

        Build the Gold review table from the Silver feature table.

        This notebook does not reread the external catalog. It reads `SUPPLY_PO_PRICE_REVIEW_FEATURES` from the standard catalog and applies deterministic review rules.

In [ ]:
# Edit these values for your AIDP workspace.
SOURCE_CATALOG = "aidp_sc_demo_source"
SOURCE_SCHEMA = "aidp_sc_demo"  # Use AIDP_SC_DEMO if your workspace exposes uppercase schema names.

TARGET_CATALOG = "aidp_sc_demo_standard"
SILVER_SCHEMA = "demo_supply_chain_silver"
GOLD_SCHEMA = "demo_supply_chain_gold"

# AIDP standard catalogs use managed Delta tables. Keep this as "delta" unless your tenancy requires a different table format.
TABLE_FORMAT = "delta"

In [ ]:
import re
from pyspark.sql import functions as F

IDENTIFIER_PATTERN = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def qident(value: str) -> str:
    """Quote and validate a catalog, schema, or table identifier."""
    if not IDENTIFIER_PATTERN.fullmatch(value):
        raise ValueError(f"Unsupported identifier: {value!r}")
    return f"`{value}`"


def qname(*parts: str) -> str:
    return ".".join(qident(part) for part in parts)


def show_small(df, n: int = 20) -> None:
    """Display a small result in notebook UI, falling back to show()."""
    try:
        display(df.limit(n))
    except NameError:
        df.show(n, truncate=False)


SOURCE_TABLES = {
    "suppliers": "aidp_sc_suppliers",
    "supplier_sites": "aidp_sc_supplier_sites",
    "item_categories": "aidp_sc_item_categories",
    "items": "aidp_sc_items",
    "po_headers": "aidp_sc_po_headers",
    "po_lines": "aidp_sc_po_lines",
    "blanket_prices": "aidp_sc_blanket_prices",
    "receipts": "aidp_sc_receipts",
    "invoice_lines": "aidp_sc_invoice_lines",
}


def source_table(key: str):
    return spark.table(qname(SOURCE_CATALOG, SOURCE_SCHEMA, SOURCE_TABLES[key]))


def normalize_table_name(table: str) -> str:
    """AIDP standard catalog table names are easiest to resolve consistently in lowercase."""
    normalized = table.lower()
    if not IDENTIFIER_PATTERN.fullmatch(normalized):
        raise ValueError(f"Unsupported table identifier: {table!r}")
    return normalized


def target_table(schema: str, table: str) -> str:
    return qname(TARGET_CATALOG, schema, normalize_table_name(table))


def ensure_schema(schema: str) -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qname(TARGET_CATALOG, schema)}")


def read_managed_table(schema: str, table: str):
    table_name = normalize_table_name(table)
    return spark.table(qname(TARGET_CATALOG, schema, table_name))


def write_managed_table(df, schema: str, table: str) -> None:
    """Overwrite one managed demo table in the AIDP standard catalog."""
    table_name = normalize_table_name(table)
    ensure_schema(schema)
    full_name = qname(TARGET_CATALOG, schema, table_name)
    spark.sql(f"DROP TABLE IF EXISTS {full_name}")
    writer = df.write.mode("overwrite").option("overwriteSchema", "true")
    if TABLE_FORMAT:
        writer = writer.format(TABLE_FORMAT)
    writer.saveAsTable(full_name)
    print(f"Wrote {full_name}")

## Read the Silver feature table

In [ ]:
features = read_managed_table(SILVER_SCHEMA, "SUPPLY_PO_PRICE_REVIEW_FEATURES")
print(f"Silver feature rows: {features.count():,}")

## Apply deterministic procurement review rules

In [ ]:
has_price_issue = F.col("price_signal").isin("HIGH", "LOW")
has_receipt_issue = F.col("receipt_signal").isin("REJECTED", "PARTIAL")
has_invoice_issue = F.col("invoice_signal").isin("PRICE_MISMATCH", "QUANTITY_MISMATCH")
needs_evidence = F.col("price_signal").isin("NOT_COMPARABLE", "NO_REFERENCE") | F.col("receipt_signal").isin("NOT_FOUND") | F.col("invoice_signal").isin("NOT_FOUND")

gold = (
    features
    .withColumn(
        "review_outcome",
        F.when(needs_evidence, F.lit("GATHER_EVIDENCE"))
        .when(has_price_issue & has_receipt_issue & has_invoice_issue, F.lit("COMBINED_PROCUREMENT_REVIEW"))
        .when(has_price_issue, F.lit("BLANKET_PRICE_REVIEW"))
        .when(has_receipt_issue, F.lit("RECEIVING_REVIEW"))
        .when(has_invoice_issue, F.lit("INVOICE_MATCH_REVIEW"))
        .otherwise(F.lit("MONITOR")),
    )
    .withColumn(
        "review_severity",
        F.when(F.col("review_outcome") == "COMBINED_PROCUREMENT_REVIEW", F.lit("HIGH"))
        .when(F.col("review_outcome").isin("BLANKET_PRICE_REVIEW", "RECEIVING_REVIEW", "INVOICE_MATCH_REVIEW"), F.lit("MEDIUM"))
        .when(F.col("review_outcome") == "GATHER_EVIDENCE", F.lit("LOW"))
        .otherwise(F.lit("INFO")),
    )
    .withColumn(
        "review_reason",
        F.concat_ws(
            "; ",
            F.when(F.col("price_signal") == "HIGH", F.lit("PO unit price is materially above the selected reference price")),
            F.when(F.col("price_signal") == "LOW", F.lit("PO unit price is materially below the selected reference price")),
            F.when(F.col("price_signal") == "NOT_COMPARABLE", F.lit("PO line is not comparable by UOM or currency")),
            F.when(F.col("price_signal") == "NO_REFERENCE", F.lit("No usable price reference was found")),
            F.when(F.col("receipt_signal") == "REJECTED", F.lit("Receipt evidence includes rejected quantity")),
            F.when(F.col("receipt_signal") == "PARTIAL", F.lit("Receipt evidence is partial")),
            F.when(F.col("receipt_signal") == "NOT_FOUND", F.lit("No receipt evidence was found")),
            F.when(F.col("invoice_signal") == "PRICE_MISMATCH", F.lit("Invoice price does not match the PO line")),
            F.when(F.col("invoice_signal") == "QUANTITY_MISMATCH", F.lit("Invoice quantity does not match the PO line")),
            F.when(F.col("invoice_signal") == "NOT_FOUND", F.lit("No invoice evidence was found")),
            F.when(F.col("review_outcome") == "MONITOR", F.lit("No review exception found")),
        ),
    )
    .withColumn("review_owner", F.when(F.col("review_outcome") == "INVOICE_MATCH_REVIEW", F.lit("Accounts Payable")).otherwise(F.lit("Buyer")))
    .withColumn("review_created_ts", F.current_timestamp())
    .select(
        "po_header_id",
        "po_line_id",
        "po_number",
        "line_number",
        "buyer_name",
        "supplier_number",
        "supplier_name",
        "supplier_site",
        "item_number",
        "item_description",
        "ordered_quantity",
        "uom_code",
        "currency_code",
        "actual_unit_price",
        "reference_unit_price",
        "reference_source",
        "price_delta_amount",
        "price_delta_percent",
        "price_signal",
        "receipt_signal",
        "invoice_signal",
        "review_outcome",
        "review_severity",
        "review_owner",
        "review_reason",
        "review_created_ts",
    )
)

## Write Gold output

In [ ]:
write_managed_table(gold, GOLD_SCHEMA, "SUPPLY_PO_PRICE_REVIEW_OUTPUT")

show_small(read_managed_table(GOLD_SCHEMA, "SUPPLY_PO_PRICE_REVIEW_OUTPUT").select(
    "po_number", "line_number", "price_signal", "receipt_signal", "invoice_signal", "review_outcome", "review_severity"
).where(F.col("po_number").startswith("PO-DEMO-")), 20)